In [1]:
import pandas as pd
import datetime
import inspect

import optuna
import optuna.visualization as vis
from optuna.trial import FrozenTrial, TrialState
from optuna.distributions import FloatDistribution, CategoricalDistribution

In [2]:
def dataframe_to_study(df, study_name="restored_study", directions=None):
    study = optuna.create_study(study_name=study_name, directions=directions)

    KNOWN_DISTRIBUTIONS = {
        'model': CategoricalDistribution([
            'custom_unet', 'monai_unet', 'attention_unet', 'dynunet'
        ]),
        'loss': CategoricalDistribution([
            'dice', 'dice_ce', 'dice_focal', 'tversky', 'generalized_dice', 'dice_ce_combo'
        ]),
        'lr': FloatDistribution(1e-5, 1e-3, log=True)
    }

    sig = inspect.signature(FrozenTrial.__init__)
    params_names = list(sig.parameters.keys())
    
    use_values = 'values' in params_names
    use_trial_id = 'trial_id' in params_names
    use_value = 'value' in params_names
    
    trial_id_counter = 0
    for _, row in df.iterrows():
        number = int(row['number'])
        state = TrialState[row['state']]
        
        dt_start = row['datetime_start']
        dt_complete = row['datetime_complete']
        datetime_start = pd.to_datetime(dt_start) if pd.notnull(dt_start) else None
        datetime_complete = pd.to_datetime(dt_complete) if pd.notnull(dt_complete) else None

        values_list = []
        for col in sorted([c for c in df.columns if c.startswith('values_')]):
            val = row[col]
            values_list.append(float(val) if pd.notnull(val) else None)
        
        if not values_list or all(v is None for v in values_list):
            values_list = None

        params = {}
        distributions = {}
        
        for col in df.columns:
            if col.startswith('params_'):
                param_name = col[len('params_'):]
                param_val = row[col]
                
                if pd.notnull(param_val):
                    params[param_name] = param_val
                    if param_name in KNOWN_DISTRIBUTIONS:
                        distributions[param_name] = KNOWN_DISTRIBUTIONS[param_name]
                    else:
                        if isinstance(param_val, (float, int)) and not isinstance(param_val, bool):
                            distributions[param_name] = FloatDistribution(-1e9, 1e9)
                        else:
                            distributions[param_name] = CategoricalDistribution([str(param_val)])

        user_attrs = {}
        system_attrs = {}
        
        for col in df.columns:
            if col.startswith('user_attrs_'):
                attr_name = col[len('user_attrs_'):]
                if pd.notnull(row[col]):
                    user_attrs[attr_name] = row[col]
            elif col.startswith('system_attrs_'):
                attr_name = col[len('system_attrs_'):]
                if pd.notnull(row[col]):
                    system_attrs[attr_name] = row[col]

        trial_kwargs = {
            'number': number,
            'datetime_start': datetime_start,
            'datetime_complete': datetime_complete,
            'params': params,
            'distributions': distributions,
            'user_attrs': user_attrs,
            'system_attrs': system_attrs,
            'state': state,
            'intermediate_values': {}
        }
        
        if use_values:
            trial_kwargs['values'] = values_list
            if use_value:
                trial_kwargs['value'] = None
        elif use_value:
            trial_kwargs['value'] = values_list[0] if values_list and len(values_list) > 0 else None
        
        if use_trial_id:
            trial_kwargs['trial_id'] = trial_id_counter
            trial_id_counter += 1

        trial = FrozenTrial(**trial_kwargs)
        study.add_trial(trial)

    return study

In [3]:
trials_df = pd.read_csv("/kaggle/input/datasets/l1ghtsource/cv-hw02-outputs/optim_logs.csv") 

In [4]:
trials_df

,number,values_0,values_1,datetime_start,datetime_complete,duration,params_loss,params_lr,params_model,user_attrs_f1,user_attrs_iou,system_attrs_NSGAIISampler:generation,state
0,0,0.784916,0.874178,2026-03-07 17:00:30.020130,2026-03-07 17:10:55.328745,0 days 00:10:25.308615,dice_ce,0.000028,monai_unet,0.874178,0.784916,0,COMPLETE
1,1,0.797193,0.881559,2026-03-07 17:10:55.329738,2026-03-07 17:22:42.924160,0 days 00:11:47.594422,dice_ce_combo,0.000091,dynunet,0.881559,0.797193,0,COMPLETE
2,2,0.793140,0.878751,2026-03-07 17:22:42.925105,2026-03-07 17:32:16.453664,0 days 00:09:33.528559,dice_ce,0.000132,monai_unet,0.878751,0.793140,0,COMPLETE
3,3,0.902203,0.945874,2026-03-07 17:32:16.454989,2026-03-07 17:48:34.522090,0 days 00:16:18.067101,dice_ce_combo,0.000299,custom_unet,0.945874,0.902203,0,COMPLETE
4,4,0.800266,0.884012,2026-03-07 17:48:34.523054,2026-03-07 17:58:10.357342,0 days 00:09:35.834288,dice_ce,0.000150,monai_unet,0.884012,0.800266,0,COMPLETE
5,5,0.781255,0.871461,2026-03-07 17:58:10.358544,2026-03-07 18:07:46.711043,0 days 00:09:36.352499,dice_ce_combo,0.000039,monai_unet,0.871461,0.781255,0,COMPLETE
6,6,0.796178,0.880914,2026-03-07 18:07:46.711940,2026-03-07 18:19:16.172850,0 days 00:11:29.460910,dice,0.000061,dynunet,0.880914,0.796178,0,COMPLETE
7,7,0.770124,0.864353,2026-03-07 18:19:16.174105,2026-03-07 18:28:47.534178,0 days 00:09:31.360073,dice,0.000014,monai_unet,0.864353,0.770124,0,COMPLETE
8,8,0.837455,0.907584,2026-03-07 18:28:47.535504,2026-03-07 18:38:24.391634,0 days 00:09:36.856130,tversky,0.000505,monai_unet,0.907584,0.837455,0,COMPLETE
9,9,0.888820,0.937961,2026-03-07 18:38:24.392769,2026-03-07 18:54:06.313063,0 days 00:15:41.920294,dice,0.000963,custom_unet,0.937961,0.888820,0,COMPLETE


In [5]:
study = dataframe_to_study(
    df=trials_df, 
    study_name="restored_multiobjective", 
    directions=['maximize', 'maximize']
)

[I 2026-03-13 09:53:41,726] A new study created in memory with name: restored_multiobjective


In [6]:
def save_plot(fig, filename):
    html_str = fig.to_html(include_plotlyjs='inline', full_html=False)
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_str)

In [7]:
save_plot(optuna.visualization.plot_pareto_front(study, target_names=['iou', 'f1']), 'plot_pareto_front.html')

In [8]:
save_plot(vis.plot_optimization_history(study, target=lambda t: t.values[0], target_name='iou'), 'plot_optimization_history_iou.html')

In [9]:
save_plot(vis.plot_optimization_history(study, target=lambda t: t.values[1], target_name='f1'), 'plot_optimization_history_f1.html')

In [10]:
save_plot(vis.plot_param_importances(study, target=lambda t: t.values[0], target_name='iou'), 'plot_param_importances_iou.html')

In [11]:
save_plot(vis.plot_param_importances(study, target=lambda t: t.values[1], target_name='f1'), 'plot_param_importances_f1.html')

In [12]:
save_plot(vis.plot_parallel_coordinate(study, target=lambda t: t.values[0], target_name='iou'), 'plot_parallel_coordinate_iou.html')

In [13]:
save_plot(vis.plot_parallel_coordinate(study, target=lambda t: t.values[1], target_name='f1'), 'plot_parallel_coordinate_f1.html')